# MedicalNLP-RAG — Phase 3A
## BioASQ 14b Training Data Audit and Frozen Internal Split

This notebook continues the PubMedQA baseline experiments.

### Input
`/content/drive/MyDrive/MedicalNLP_RAG/training14b.json`

### Goals
1. Read the actual BioASQ 14b JSON schema without assuming field completeness.
2. Validate question IDs and core fields.
3. Quantify question types, gold documents, snippets, exact answers, ideal answers, and concepts.
4. Detect exact and normalized duplicate questions.
5. Extract PubMed IDs from document/snippet URLs.
6. Build a deterministic stratified internal split:
   - train: 70%
   - validation: 15%
   - internal test: 15%
7. Freeze the internal test IDs immediately.
8. Export clean CSV/JSON tables and 300-dpi figures.

### Important
This is **not** the official BioASQ challenge test set.
It is a frozen internal hold-out created from `training14b.json` for our research experiments.

No LLM, RAG, BM25 or dense retrieval is run in this notebook.

## 0. Configuration

In [ ]:
PROJECT_NAME = "MedicalNLP_RAG"
BIOASQ_FILENAME = "training14b.json"

SEED = 42

TRAIN_FRAC = 0.70
VAL_FRAC = 0.15
TEST_FRAC = 0.15

assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Imports and project paths

In [ ]:
from pathlib import Path
import json
import re
import hashlib
import random
import unicodedata
import platform
import sys
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

random.seed(SEED)
np.random.seed(SEED)

ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
BIOASQ_PATH = ROOT / BIOASQ_FILENAME

PHASE3_DIR = ROOT / "results" / "phase3_bioasq14"
TABLE_DIR = PHASE3_DIR / "tables"
FIG_DIR = PHASE3_DIR / "figures"
SPLIT_DIR = ROOT / "data" / "bioasq14_splits"
PROC_DIR = ROOT / "data" / "bioasq14_processed"
LOG_DIR = PHASE3_DIR / "logs"

for p in [PHASE3_DIR, TABLE_DIR, FIG_DIR, SPLIT_DIR, PROC_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

assert BIOASQ_PATH.exists(), f"BioASQ file not found: {BIOASQ_PATH}"

print("BioASQ file:", BIOASQ_PATH)
print("Size (MB):", round(BIOASQ_PATH.stat().st_size / 1024**2, 2))
print("Output:", PHASE3_DIR)

## 3. Record file hash and environment

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

file_hash = sha256_file(BIOASQ_PATH)

environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
    "bioasq_file": str(BIOASQ_PATH),
    "bioasq_sha256": file_hash,
}

with open(LOG_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)

print(json.dumps(environment, indent=2, ensure_ascii=False))

## 4. Load JSON and inspect the real top-level schema

In [ ]:
with open(BIOASQ_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

print("Top-level Python type:", type(raw).__name__)

if isinstance(raw, dict):
    print("Top-level keys:", list(raw.keys())[:30])
elif isinstance(raw, list):
    print("Top-level list length:", len(raw))

## 5. Resolve the question list robustly

In [ ]:
if isinstance(raw, dict) and isinstance(raw.get("questions"), list):
    questions = raw["questions"]
    QUESTION_CONTAINER = "questions"
elif isinstance(raw, list):
    questions = raw
    QUESTION_CONTAINER = "top_level_list"
else:
    candidate_keys = []
    if isinstance(raw, dict):
        for k, v in raw.items():
            if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                candidate_keys.append(k)

    raise ValueError(
        "Could not automatically locate the question list. "
        f"Candidate list-valued keys: {candidate_keys}"
    )

print("Question container:", QUESTION_CONTAINER)
print("Number of questions:", len(questions))

assert len(questions) > 0
assert isinstance(questions[0], dict)

print("\nFirst question keys:")
print(sorted(questions[0].keys()))

## 6. Field-presence audit

In [ ]:
all_keys = sorted(set().union(*(q.keys() for q in questions)))

field_rows = []
for key in all_keys:
    present = sum(key in q for q in questions)
    nonempty = 0

    for q in questions:
        value = q.get(key)
        if value is None:
            continue
        if isinstance(value, (list, dict, str)) and len(value) == 0:
            continue
        nonempty += 1

    field_rows.append({
        "field": key,
        "present_n": present,
        "present_pct": present / len(questions),
        "nonempty_n": nonempty,
        "nonempty_pct": nonempty / len(questions),
    })

field_audit = pd.DataFrame(field_rows).sort_values(
    ["present_n", "field"],
    ascending=[False, True]
)

field_audit.to_csv(TABLE_DIR / "field_presence_audit.csv", index=False)
display(field_audit)

## 7. Normalize the dataset to one row per question

In [ ]:
def ensure_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    return [value]

def text_or_empty(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    return str(value).strip()

def count_exact_answers(value):
    if value is None:
        return 0
    if isinstance(value, str):
        return 1 if value.strip() else 0
    if isinstance(value, list):
        return len(value)
    return 1

rows = []

for i, q in enumerate(questions):
    qid = text_or_empty(q.get("id"))
    body = text_or_empty(q.get("body"))
    qtype = text_or_empty(q.get("type")).lower()

    documents = ensure_list(q.get("documents"))
    snippets = ensure_list(q.get("snippets"))
    concepts = ensure_list(q.get("concepts"))
    ideal_answer = ensure_list(q.get("ideal_answer"))
    exact_answer = q.get("exact_answer")

    rows.append({
        "row_index": i,
        "id": qid,
        "type": qtype,
        "body": body,
        "n_documents": len(documents),
        "n_snippets": len(snippets),
        "n_concepts": len(concepts),
        "n_exact_answer_items": count_exact_answers(exact_answer),
        "n_ideal_answers": len(ideal_answer),
        "has_documents": len(documents) > 0,
        "has_snippets": len(snippets) > 0,
        "has_exact_answer": exact_answer not in [None, "", []],
        "has_ideal_answer": len(ideal_answer) > 0,
    })

df = pd.DataFrame(rows)

print("Shape:", df.shape)
display(df.head())

## 8. Integrity checks

In [ ]:
integrity = {
    "n_questions": int(len(df)),
    "unique_ids": int(df["id"].nunique()),
    "duplicate_ids": int(df["id"].duplicated().sum()),
    "empty_ids": int((df["id"].str.len() == 0).sum()),
    "empty_bodies": int((df["body"].str.len() == 0).sum()),
    "empty_types": int((df["type"].str.len() == 0).sum()),
    "question_types": {
        str(k): int(v)
        for k, v in df["type"].value_counts(dropna=False).to_dict().items()
    },
    "questions_with_documents": int(df["has_documents"].sum()),
    "questions_with_snippets": int(df["has_snippets"].sum()),
    "questions_with_exact_answer": int(df["has_exact_answer"].sum()),
    "questions_with_ideal_answer": int(df["has_ideal_answer"].sum()),
}

with open(TABLE_DIR / "integrity_summary.json", "w", encoding="utf-8") as f:
    json.dump(integrity, f, indent=2, ensure_ascii=False)

print(json.dumps(integrity, indent=2, ensure_ascii=False))

## 9. Question-type distribution

In [ ]:
type_counts = (
    df["type"]
    .value_counts()
    .rename_axis("type")
    .reset_index(name="n")
)

type_counts["pct"] = type_counts["n"] / len(df)

type_counts.to_csv(TABLE_DIR / "question_type_distribution.csv", index=False)
display(type_counts)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(type_counts["type"], type_counts["n"])
ax.set_title("BioASQ 14b training question-type distribution")
ax.set_xlabel("Question type")
ax.set_ylabel("Number of questions")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()

fig.savefig(
    FIG_DIR / "fig01_question_type_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 10. Gold-evidence availability by question type

In [ ]:
evidence_by_type = (
    df.groupby("type")
    .agg(
        n=("id", "size"),
        mean_documents=("n_documents", "mean"),
        median_documents=("n_documents", "median"),
        mean_snippets=("n_snippets", "mean"),
        median_snippets=("n_snippets", "median"),
        pct_has_documents=("has_documents", "mean"),
        pct_has_snippets=("has_snippets", "mean"),
        pct_has_exact_answer=("has_exact_answer", "mean"),
        pct_has_ideal_answer=("has_ideal_answer", "mean"),
    )
    .round(4)
    .reset_index()
)

evidence_by_type.to_csv(TABLE_DIR / "evidence_availability_by_type.csv", index=False)
display(evidence_by_type)

## 11. Question text length analysis

In [ ]:
df["body_chars"] = df["body"].str.len()
df["body_words"] = df["body"].str.split().str.len()

length_summary = (
    df.groupby("type")[["body_chars", "body_words"]]
    .describe()
    .round(2)
)

length_summary.to_csv(TABLE_DIR / "question_length_summary.csv")
display(length_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for qtype, group in df.groupby("type"):
    ax.hist(group["body_words"], bins=35, alpha=0.45, label=qtype)

ax.set_title("Question length distribution")
ax.set_xlabel("Question length (words)")
ax.set_ylabel("Frequency")
ax.legend()
fig.tight_layout()

fig.savefig(
    FIG_DIR / "fig02_question_length_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 12. Exact and normalized duplicate-question audit

In [ ]:
def normalize_question(text):
    text = unicodedata.normalize("NFKC", str(text))
    text = text.lower().strip()
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["body_normalized"] = df["body"].map(normalize_question)

exact_dup_mask = df["body"].duplicated(keep=False) & (df["body"].str.len() > 0)
norm_dup_mask = (
    df["body_normalized"].duplicated(keep=False)
    & (df["body_normalized"].str.len() > 0)
)

duplicate_summary = {
    "exact_duplicate_rows": int(exact_dup_mask.sum()),
    "exact_duplicate_unique_texts": int(df.loc[exact_dup_mask, "body"].nunique()),
    "normalized_duplicate_rows": int(norm_dup_mask.sum()),
    "normalized_duplicate_unique_texts": int(
        df.loc[norm_dup_mask, "body_normalized"].nunique()
    ),
}

with open(
    TABLE_DIR / "duplicate_question_summary.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(duplicate_summary, f, indent=2, ensure_ascii=False)

print(json.dumps(duplicate_summary, indent=2))

duplicate_rows = df.loc[
    norm_dup_mask,
    ["id", "type", "body", "body_normalized"]
].sort_values("body_normalized")

duplicate_rows.to_csv(
    TABLE_DIR / "normalized_duplicate_questions.csv",
    index=False
)

display(duplicate_rows.head(30))

## 13. Extract PubMed IDs from gold document URLs

In [ ]:
def extract_numeric_id_from_reference(ref):
    if ref is None:
        return None

    s = str(ref).strip().rstrip("/")
    if not s:
        return None

    match = re.search(r"(\d+)(?:\D*)$", s)
    return match.group(1) if match else None

document_rows = []
snippet_rows = []

for q in questions:
    qid = text_or_empty(q.get("id"))
    qtype = text_or_empty(q.get("type")).lower()

    for rank, doc in enumerate(ensure_list(q.get("documents")), start=1):
        ref = str(doc).strip()
        document_rows.append({
            "question_id": qid,
            "question_type": qtype,
            "rank_in_gold": rank,
            "document_reference": ref,
            "pubmed_id": extract_numeric_id_from_reference(ref),
        })

    for rank, snip in enumerate(ensure_list(q.get("snippets")), start=1):
        if isinstance(snip, dict):
            doc_ref = text_or_empty(snip.get("document"))
            text = text_or_empty(snip.get("text"))
            begin_section = text_or_empty(snip.get("beginSection"))
            end_section = text_or_empty(snip.get("endSection"))
            begin_offset = snip.get("offsetInBeginSection")
            end_offset = snip.get("offsetInEndSection")
        else:
            doc_ref = ""
            text = str(snip)
            begin_section = ""
            end_section = ""
            begin_offset = None
            end_offset = None

        snippet_rows.append({
            "question_id": qid,
            "question_type": qtype,
            "snippet_rank": rank,
            "document_reference": doc_ref,
            "pubmed_id": extract_numeric_id_from_reference(doc_ref),
            "snippet_text": text,
            "begin_section": begin_section,
            "end_section": end_section,
            "begin_offset": begin_offset,
            "end_offset": end_offset,
        })

documents_df = pd.DataFrame(document_rows)
snippets_df = pd.DataFrame(snippet_rows)

documents_df.to_csv(PROC_DIR / "bioasq14_gold_documents.csv", index=False)
snippets_df.to_csv(PROC_DIR / "bioasq14_gold_snippets.csv", index=False)

print("Gold document rows:", len(documents_df))
print("Unique parsed PubMed IDs:", documents_df["pubmed_id"].dropna().nunique())
print("Unparsed document references:", documents_df["pubmed_id"].isna().sum())
print()
print("Snippet rows:", len(snippets_df))
print("Snippet-linked unique PubMed IDs:", snippets_df["pubmed_id"].dropna().nunique())

## 14. Gold evidence statistics

In [ ]:
gold_stats = {
    "document_rows": int(len(documents_df)),
    "unique_document_references": int(
        documents_df["document_reference"].nunique()
    ) if len(documents_df) else 0,
    "unique_parsed_pubmed_ids": int(
        documents_df["pubmed_id"].dropna().nunique()
    ) if len(documents_df) else 0,
    "unparsed_document_references": int(
        documents_df["pubmed_id"].isna().sum()
    ) if len(documents_df) else 0,
    "snippet_rows": int(len(snippets_df)),
    "unique_snippet_texts": int(
        snippets_df["snippet_text"].nunique()
    ) if len(snippets_df) else 0,
    "empty_snippet_texts": int(
        (snippets_df["snippet_text"].str.len() == 0).sum()
    ) if len(snippets_df) else 0,
}

with open(TABLE_DIR / "gold_evidence_summary.json", "w", encoding="utf-8") as f:
    json.dump(gold_stats, f, indent=2, ensure_ascii=False)

print(json.dumps(gold_stats, indent=2))

# 15. Create a frozen internal split

Normalized duplicate questions are assigned to the same group before splitting.
This prevents normalized duplicates from appearing on opposite sides of the split.

In [ ]:
df["group_key"] = np.where(
    df["body_normalized"].str.len() > 0,
    df["body_normalized"],
    df["id"]
)

group_table = (
    df.groupby("group_key")
    .agg(
        representative_type=(
            "type",
            lambda x: x.mode().iloc[0] if not x.mode().empty else x.iloc[0]
        ),
        n_questions=("id", "size")
    )
    .reset_index()
)

print("Questions:", len(df))
print("Duplicate-safe groups:", len(group_table))
display(group_table["n_questions"].value_counts().sort_index().head(20))

## 16. Stratified split at duplicate-group level

In [ ]:
def safe_split_groups(group_table, seed=SEED):
    stratify_used_first = True
    stratify_used_second = True

    y = group_table["representative_type"]

    try:
        train_g, temp_g = train_test_split(
            group_table,
            test_size=(VAL_FRAC + TEST_FRAC),
            random_state=seed,
            stratify=y
        )
    except ValueError as e:
        print("First stratified split failed:", e)
        stratify_used_first = False
        train_g, temp_g = train_test_split(
            group_table,
            test_size=(VAL_FRAC + TEST_FRAC),
            random_state=seed,
            shuffle=True
        )

    relative_test = TEST_FRAC / (VAL_FRAC + TEST_FRAC)

    try:
        val_g, test_g = train_test_split(
            temp_g,
            test_size=relative_test,
            random_state=seed,
            stratify=temp_g["representative_type"]
        )
    except ValueError as e:
        print("Second stratified split failed:", e)
        stratify_used_second = False
        val_g, test_g = train_test_split(
            temp_g,
            test_size=relative_test,
            random_state=seed,
            shuffle=True
        )

    return train_g, val_g, test_g, stratify_used_first, stratify_used_second

train_g, val_g, test_g, strat1, strat2 = safe_split_groups(group_table)

group_to_split = {}
for g in train_g["group_key"]:
    group_to_split[g] = "train"
for g in val_g["group_key"]:
    group_to_split[g] = "validation"
for g in test_g["group_key"]:
    group_to_split[g] = "internal_test_LOCKED"

df["split"] = df["group_key"].map(group_to_split)

assert df["split"].notna().all()

split_counts = (
    df.groupby(["split", "type"])
    .size()
    .unstack(fill_value=0)
)

split_counts["TOTAL"] = split_counts.sum(axis=1)
display(split_counts)

## 17. Verify no normalized-question leakage across splits

In [ ]:
group_split_counts = df.groupby("group_key")["split"].nunique()
n_cross_split_groups = int((group_split_counts > 1).sum())

split_integrity = {
    "n_total_questions": int(len(df)),
    "train_n": int((df["split"] == "train").sum()),
    "validation_n": int((df["split"] == "validation").sum()),
    "internal_test_n": int((df["split"] == "internal_test_LOCKED").sum()),
    "duplicate_safe_groups": int(len(group_table)),
    "cross_split_duplicate_groups": n_cross_split_groups,
    "first_split_stratified": bool(strat1),
    "second_split_stratified": bool(strat2),
    "seed": SEED,
}

with open(
    SPLIT_DIR / "split_integrity.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(split_integrity, f, indent=2, ensure_ascii=False)

print(json.dumps(split_integrity, indent=2))
assert n_cross_split_groups == 0

## 18. Freeze the split files

In [ ]:
split_manifest = df[
    ["id", "type", "body", "body_normalized", "group_key", "split"]
].copy()

split_manifest.to_csv(
    SPLIT_DIR / "bioasq14_internal_split_manifest.csv",
    index=False
)

for split_name in ["train", "validation", "internal_test_LOCKED"]:
    ids = (
        df.loc[df["split"] == split_name, "id"]
        .astype(str)
        .tolist()
    )

    with open(
        SPLIT_DIR / f"{split_name}_ids.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(ids, f, indent=2, ensure_ascii=False)

print("Frozen split files saved to:", SPLIT_DIR)

## 19. Export the original question objects by split

In [ ]:
id_to_split = dict(zip(df["id"], df["split"]))

split_objects = {
    "train": [],
    "validation": [],
    "internal_test_LOCKED": []
}

for q in questions:
    qid = text_or_empty(q.get("id"))
    split_objects[id_to_split[qid]].append(q)

for split_name, qs in split_objects.items():
    payload = {"questions": qs}

    with open(
        SPLIT_DIR / f"{split_name}.json",
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(split_name, len(qs))

## 20. Split-distribution figure

In [ ]:
plot_df = (
    df.groupby(["split", "type"])
    .size()
    .reset_index(name="n")
)

pivot = plot_df.pivot(
    index="type",
    columns="split",
    values="n"
).fillna(0)

fig, ax = plt.subplots(figsize=(8, 5))
pivot.plot(kind="bar", ax=ax)
ax.set_title("BioASQ 14b internal split by question type")
ax.set_xlabel("Question type")
ax.set_ylabel("Number of questions")
ax.tick_params(axis="x", rotation=0)
fig.tight_layout()

fig.savefig(
    FIG_DIR / "fig03_internal_split_distribution.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

## 21. Document overlap audit across splits

In [ ]:
qid_split = dict(zip(df["id"], df["split"]))

doc_split_sets = {
    "train": set(),
    "validation": set(),
    "internal_test_LOCKED": set()
}

for row in document_rows:
    split_name = qid_split[row["question_id"]]
    ref = row["document_reference"]
    if ref:
        doc_split_sets[split_name].add(ref)

overlap_rows = []

split_names = list(doc_split_sets.keys())
for i in range(len(split_names)):
    for j in range(i + 1, len(split_names)):
        a = split_names[i]
        b = split_names[j]

        inter = doc_split_sets[a] & doc_split_sets[b]
        union = doc_split_sets[a] | doc_split_sets[b]

        overlap_rows.append({
            "split_a": a,
            "split_b": b,
            "docs_a": len(doc_split_sets[a]),
            "docs_b": len(doc_split_sets[b]),
            "overlap_n": len(inter),
            "jaccard": len(inter) / len(union) if union else 0.0
        })

document_overlap_df = pd.DataFrame(overlap_rows)
document_overlap_df.to_csv(
    TABLE_DIR / "document_overlap_across_splits.csv",
    index=False
)
display(document_overlap_df)

## 22. Create Phase 3A summary report

In [ ]:
phase3_report = {
    "dataset_file": str(BIOASQ_PATH),
    "sha256": file_hash,
    "question_container": QUESTION_CONTAINER,
    "n_questions": int(len(df)),
    "field_names": all_keys,
    "integrity": integrity,
    "duplicate_summary": duplicate_summary,
    "gold_evidence": gold_stats,
    "split_integrity": split_integrity,
    "split_type_counts": (
        df.groupby(["split", "type"])
        .size()
        .reset_index(name="n")
        .to_dict(orient="records")
    ),
}

with open(
    PHASE3_DIR / "phase3A_report.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        phase3_report,
        f,
        indent=2,
        ensure_ascii=False
    )

print(json.dumps(phase3_report, indent=2, ensure_ascii=False))

# 23. STOP POINT 3A

Stop here and send:

1. Number of questions + first-question keys
2. `field_presence_audit`
3. `integrity_summary`
4. `question_type_distribution`
5. `evidence_availability_by_type`
6. `duplicate_question_summary`
7. `gold_evidence_summary`
8. internal split table
9. `document_overlap_across_splits`
10. `phase3A_report.json`

### Do not start retrieval yet.

After review, Phase 3B will build:
- BM25 baseline
- biomedical dense retriever
- hybrid retrieval / RRF
- Recall@K, Precision@K, MAP@K, MRR
- gold-document and gold-snippet analysis

The internal test remains locked while retrieval methods are selected on validation.

In [ ]:
print("PHASE 3A COMPLETE")
print("Results:", PHASE3_DIR)
print("Splits :", SPLIT_DIR)

print("\nCreated files:")
for p in sorted(PHASE3_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(ROOT))

for p in sorted(SPLIT_DIR.rglob("*")):
    if p.is_file():
        print(" -", p.relative_to(ROOT))

In [ ]:
aux_rows = []

for q in questions:
    if (
        "_body" in q
        or "_type" in q
        or "duplicate_tmp" in q
    ):
        aux_rows.append({
            "id": str(q.get("id", "")),
            "type": q.get("type"),
            "_type": q.get("_type"),
            "body": q.get("body"),
            "_body": q.get("_body"),
            "duplicate_tmp": q.get("duplicate_tmp"),
        })

aux_df = pd.DataFrame(aux_rows)

print("Records with auxiliary fields:", len(aux_df))

print("\nField counts:")
print(aux_df.notna().sum())

if len(aux_df):
    aux_df["body_same"] = (
        aux_df["body"].fillna("").astype(str).str.strip()
        ==
        aux_df["_body"].fillna("").astype(str).str.strip()
    )

    aux_df["type_same"] = (
        aux_df["type"].fillna("").astype(str).str.strip()
        ==
        aux_df["_type"].fillna("").astype(str).str.strip()
    )

    print("\nbody == _body:")
    print(aux_df["body_same"].value_counts(dropna=False))

    print("\ntype == _type:")
    print(aux_df["type_same"].value_counts(dropna=False))

display(aux_df)

aux_df.to_csv(
    TABLE_DIR / "auxiliary_field_audit.csv",
    index=False
)